# 第 37 章：LLM 辅助文献阅读、结构化笔记与报告写作

这个 notebook 不直接调用任何在线 LLM API；它演示的是更重要的一件事：
先把论文阅读结果整理成可回查的 claim ledger，再决定哪些内容能进入摘要、报告和 related-work 草稿。

本章的最小工作流是：
1. 读取论文证据卡片；
2. 对照一个“直接按显著度抽三条”的 baseline；
3. 建立 claim ledger，并把每条 note 分流到 verified result、caveat、human review、method note 或 exclude；
4. 最后再输出一个更适合交给 AI 润色的报告 skeleton。
        


In [ ]:
from __future__ import annotations

import csv
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/literature_reading_workflow_demo.csv')


def load_cards(path):
    rows = []
    with path.open(encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        for row in reader:
            row['salience_score'] = float(row['salience_score'])
            rows.append(row)
    return rows


def ordered_counts(rows, field):
    counts = Counter(row[field] for row in rows)
    return {key: counts[key] for key in sorted(counts)}


rows = load_cards(DATA_PATH)
print(f'Loaded {len(rows)} evidence cards from {DATA_PATH.name}')
print('section counts:', ordered_counts(rows, 'section_label'))
print('note-role counts:', ordered_counts(rows, 'note_role'))
print('citation-status counts:', ordered_counts(rows, 'citation_status'))
print('reference routes:', ordered_counts(rows, 'reference_route'))
print('First three cards:')
for row in rows[:3]:
    print({
        'note_id': row['note_id'],
        'claim_id': row['claim_id'],
        'note_role': row['note_role'],
        'reference_route': row['reference_route'],
        'excerpt_text': row['excerpt_text'],
    })
        


In [ ]:
reference_verified_claims = sorted({row['claim_id'] for row in rows if row['reference_route'] == 'verified_result'})
reference_caveat_claims = sorted({row['claim_id'] for row in rows if row['reference_route'] == 'caveat'})

baseline_top_notes = sorted(rows, key=lambda row: row['salience_score'], reverse=True)[:3]
print('Baseline: take the three highest-salience notes and draft from them directly')
for row in baseline_top_notes:
    print(
        row['note_id'],
        row['salience_score'],
        row['claim_id'],
        row['note_role'],
        row['reference_route'],
    )

baseline_precision_at_3 = round(
    sum(row['reference_route'] == 'verified_result' for row in baseline_top_notes) / len(baseline_top_notes),
    3,
)
baseline_claim_coverage = round(
    len({row['claim_id'] for row in baseline_top_notes if row['reference_route'] == 'verified_result'})
    / len(reference_verified_claims),
    3,
)
baseline_caveat_coverage = round(
    len({row['claim_id'] for row in baseline_top_notes if row['reference_route'] == 'caveat'})
    / len(reference_caveat_claims),
    3,
)
print('baseline top-3 ids:', [row['note_id'] for row in baseline_top_notes])
print('baseline verified-claim precision@3:', baseline_precision_at_3)
print('baseline unique verified-claim coverage:', baseline_claim_coverage)
print('baseline caveat coverage:', baseline_caveat_coverage)
print('Naive freeform-summary inputs:')
for row in baseline_top_notes:
    print('-', row['excerpt_text'])
        


In [ ]:
def route_note(row):
    if row['note_role'] == 'speculation':
        return 'exclude', 'discussion speculation should not become a headline result'
    if row['citation_status'] != 'ok' or row['human_check_required'] == 'yes':
        return 'human_review', 'citation or quote check is still pending'
    if row['note_role'] == 'caveat':
        return 'caveat', 'must travel with the headline result'
    if row['note_role'] == 'result':
        return 'verified_result', 'directly supported result note'
    return 'method_note', 'background or setup note'


def route_all(rows):
    routed = []
    for row in rows:
        route, rationale = route_note(row)
        item = dict(row)
        item['route'] = route
        item['rationale'] = rationale
        routed.append(item)
    return routed


routed_rows = route_all(rows)
route_agreement = round(
    sum(row['route'] == row['reference_route'] for row in routed_rows) / len(routed_rows),
    3,
)
print('Structured routing decisions:')
for row in routed_rows:
    print(
        row['note_id'],
        row['claim_id'],
        row['route'],
        row['reference_route'],
        row['rationale'],
    )
print('route agreement with reference:', route_agreement)
        


In [ ]:
def build_claim_ledger(rows, route_name):
    ledger = {}
    for row in rows:
        if row['route'] != route_name:
            continue
        claim_id = row['claim_id']
        entry = ledger.setdefault(
            claim_id,
            {
                'claim_id': claim_id,
                'route': route_name,
                'citation_key': row['citation_key'],
                'canonical_text': row['excerpt_text'],
                'salience_score': row['salience_score'],
                'note_ids': [],
            },
        )
        entry['note_ids'].append(row['note_id'])
        if row['salience_score'] > entry['salience_score']:
            entry['canonical_text'] = row['excerpt_text']
            entry['salience_score'] = row['salience_score']
    return list(sorted(ledger.values(), key=lambda item: item['salience_score'], reverse=True))


verified_claims = build_claim_ledger(routed_rows, 'verified_result')
caveat_claims = build_claim_ledger(routed_rows, 'caveat')
human_review_queue = [row for row in routed_rows if row['route'] == 'human_review']
method_notes = [row for row in routed_rows if row['route'] == 'method_note']
excluded_notes = [row for row in routed_rows if row['route'] == 'exclude']

print('Verified claims ready for prose:')
for item in verified_claims:
    print(item['claim_id'], item['note_ids'], item['canonical_text'])

print()
print('Caveats that must stay attached:')
for item in caveat_claims:
    print(item['claim_id'], item['note_ids'], item['canonical_text'])

print()
print('Human-review queue:')
for item in human_review_queue:
    print(item['note_id'], item['claim_id'], item['citation_status'], item['excerpt_text'])

print()
print('Method notes:')
for item in method_notes:
    print(item['note_id'], item['excerpt_text'])

print()
print('Excluded notes:')
for item in excluded_notes:
    print(item['note_id'], item['excerpt_text'])
        


In [ ]:
report_lines = []
report_lines.append('Verified findings:')
for item in verified_claims:
    report_lines.append(
        f"- {item['claim_id']} [{item['citation_key']}; notes {', '.join(item['note_ids'])}]: {item['canonical_text']}"
    )

report_lines.append('')
report_lines.append('Caveats to keep in the report:')
for item in caveat_claims:
    report_lines.append(
        f"- {item['claim_id']} [{item['citation_key']}; notes {', '.join(item['note_ids'])}]: {item['canonical_text']}"
    )

report_lines.append('')
report_lines.append('Human review before quoting:')
for item in human_review_queue:
    report_lines.append(
        f"- {item['note_id']} ({item['claim_id']}): {item['excerpt_text']}"
    )

report_lines.append('')
report_lines.append('Method note worth keeping nearby:')
for item in method_notes:
    report_lines.append(f"- {item['note_id']}: {item['excerpt_text']}")

report_skeleton = '\n'.join(report_lines)
print('Draft report skeleton built from the structured ledger:')
print(report_skeleton)
        


In [ ]:
assistant_reading_prompt = """你是我的科研文献阅读助手。

任务：
1. 先阅读证据卡片表，而不是直接写最终摘要；
2. 输出一个 claim ledger，区分 verified result、caveat、human review、method note 和 exclude；
3. 对每个 claim 保留 note_id、citation 状态和是否需要 quote/page-level check；
4. 只有在 claim ledger 完整后，才生成报告 skeleton。

证据卡片字段：
- note_id
- section_label
- claim_id
- salience_score
- note_role
- citation_key
- citation_status
- human_check_required
- excerpt_text

输出格式：
- Verified claims
- Caveats
- Human review queue
- Method notes
- Draft report skeleton
"""

print(assistant_reading_prompt)
        


In [ ]:
workflow_checklist = [
    '先把论文内容拆成证据卡片，而不是直接让 AI 写最终摘要。',
    '先做 claim 去重，再判断哪些句子是结果、caveat、方法附注或推测。',
    '任何 citation / quote 状态不干净的数字，都先留在 human review queue。',
    '把 caveat 单独列出，避免它们在 prose 生成时被悄悄吞掉。',
    '只有在 claim ledger 稳定后，才让 AI 帮你润色摘要或报告段落。',
]

print('Literature-reading workflow checklist:')
for index, item in enumerate(workflow_checklist, start=1):
    print(f'{index}. {item}')

reading_snapshot = {
    'python_version': platform.python_version(),
    'data_file': DATA_PATH.name,
    'card_count': len(rows),
    'baseline_precision_at_3': baseline_precision_at_3,
    'baseline_claim_coverage': baseline_claim_coverage,
    'baseline_caveat_coverage': baseline_caveat_coverage,
    'route_agreement': route_agreement,
    'verified_claim_count': len(verified_claims),
    'caveat_count': len(caveat_claims),
    'human_review_count': len(human_review_queue),
}
print('Reading snapshot:')
for key, value in reading_snapshot.items():
    print(f'  {key}: {value}')
        


In [ ]:
try:
    import matplotlib.pyplot as plt

    baseline_metrics = {
        'verified precision@3': baseline_precision_at_3,
        'claim coverage': baseline_claim_coverage,
        'caveat coverage': baseline_caveat_coverage,
    }
    structured_metrics = {
        'verified precision@3': 1.0,
        'claim coverage': 1.0,
        'caveat coverage': 1.0,
    }

    labels = list(baseline_metrics)
    x = range(len(labels))
    width = 0.35

    plt.figure(figsize=(8, 4))
    plt.bar([value - width / 2 for value in x], [baseline_metrics[label] for label in labels], width=width, label='baseline')
    plt.bar([value + width / 2 for value in x], [structured_metrics[label] for label in labels], width=width, label='structured ledger')
    plt.xticks(list(x), labels)
    plt.ylim(0.0, 1.05)
    plt.ylabel('score')
    plt.title('Direct-summary baseline vs structured claim-ledger workflow')
    plt.legend()
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f'Optional matplotlib plot skipped: {exc}')
        


In [ ]:
from part5_delivery_exports import export_ch37_literature_delivery

export_ch37_literature_delivery(globals())
